# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [11]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


# Shared style for every figure
ACCENT = '#D55E00'      # vermillion (Okabe-Ito, CVD-safe) for the key group
GREY = '#BDBDBD'        # context
DARK = '#4D4D4D'
# Okabe-Ito palette: safe for all common colour-vision deficiencies (no red/green pairing)
CONT_COLORS = {'Africa': '#D55E00', 'Asia': '#0072B2', 'Europe': '#009E73',
               'North America': '#CC79A7', 'South America': '#E69F00', 'Oceania': '#56B4E9'}
FONT = dict(family='Arial', size=13, color=DARK)

(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.
*YOUR ANSWER HERE*

The question is about change over time for six continents, and lines show trends the best. I index every continent to its 1970 size (1970 = 1×) so continents of very different sizes can be compared on growth, not absolute population. Africa, the fastest grower, is the only coloured line; the others are grey context. Direct labels at the line ends replace the legend, and a dotted line marks the 1970 level.

In [12]:
# Q1
cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
base = cont[cont['Year'] == 1970].set_index('Continent')['Population']
cont['Growth'] = cont['Population'] / cont['Continent'].map(base)      # 1970 = 1x

end = cont[cont['Year'] == 2022].set_index('Continent')['Growth'].sort_values(ascending=False)
leader, second = end.index[0], end.index[1]

fig = px.line(cont, x='Year', y='Growth', color='Continent',
              color_discrete_map={c: (ACCENT if c == leader else GREY) for c in end.index},
              template='plotly_white', height=520)
fig.update_traces(line_width=2)
fig.update_traces(selector=dict(name=leader), line_width=4)

# Direct labels at the end of each line (nudged apart where lines end close together)
label_y, last = {}, None
for c, v in end.sort_values().items():
    y = v if last is None else max(v, last + 0.13)
    label_y[c], last = y, y
for c, v in end.items():
    fig.add_annotation(x=2022, y=label_y[c], text=f'<b>{c}</b> {v:.1f}×' if c == leader else f'{c} {v:.1f}×',
                       showarrow=False, xanchor='left', xshift=6,
                       font=dict(color=ACCENT if c == leader else '#808080', size=13))

fig.add_hline(y=1, line_dash='dot', line_color='#9E9E9E')
fig.update_layout(
    title=dict(text=f"{leader} is growing fastest: its population is {end[leader]:.1f}× its 1970 size,<br>"
                    f"far ahead of {second} ({end[second]:.1f}×); Europe has barely grown ({end['Europe']:.1f}×)"),
    font=FONT, showlegend=False, margin=dict(r=150))
fig.update_xaxes(title=None, showgrid=False, range=[1968, 2023],
                 tickvals=[1970, 1980, 1990, 2000, 2010, 2020])
fig.update_yaxes(title='Population relative to 1970', ticksuffix='×', rangemode='tozero')
fig.show()

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

I chose sorted horizontal bar chart because the question asked to ranks 15 countries by numbers, and bars sorted from largest to smallest make that ranking easy to read. All the bars start at zero, so lengths can be compared with our eyes and horizontal bars keep the country names readable. Colour answers the second part of the question: African countries are vermillion and the rest are grey, so you can count them easily with just a look. Value labels replace the x-axis.

In [13]:
# Q2
# YOUR CODE HERE

df['Added'] = df['2022 Population'] - df['1970 Population']
top15 = df.nlargest(15, 'Added').sort_values('Added')          # ascending -> largest at top
top15['Group'] = np.where(top15['Continent'] == 'Africa', 'Africa', 'Other')
n_africa = (top15['Group'] == 'Africa').sum()
top_af = top15[top15['Group'] == 'Africa'].iloc[-1]

fig = px.bar(top15, x='Added', y='Country', orientation='h', color='Group',
             color_discrete_map={'Africa': ACCENT, 'Other': GREY},
             text=top15['Added'].map(lambda v: f'+{v/1e6:,.0f}M'),
             template='plotly_white', height=560)
fig.update_traces(textposition='outside', cliponaxis=False, marker_line_width=0)
fig.update_yaxes(categoryorder='total ascending')                # keep sorted: largest at top
fig.update_layout(
    title=dict(text=f"<span style='color:{ACCENT}'><b>{n_africa} of the 15</b></span> countries that added the most people since 1970 are "
                    f"<span style='color:{ACCENT}'><b>African</b></span>,<br>led by {top_af['Country']} (+{top_af['Added']/1e6:.0f}M); "
                    f"India and China alone added {(top15.nlargest(2, 'Added')['Added'].sum())/1e9:.2f} billion"),
    font=FONT, showlegend=False, margin=dict(r=60), bargap=0.25)
fig.update_xaxes(visible=False, range=[0, top15['Added'].max() * 1.12])
fig.update_yaxes(title=None, showgrid=False)
fig.show()

---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

I chose the scatter plot because the question asks about the relationship between two numeric variables, area and population, and a scatter plot can show the result the best. Both axes use a log scale because both variables range from tiny islands to giant countries. If we use a linear scale almost every point would stay in one corner. A grey dashed trend line shows the general pattern. Colour encodes continent with a CVD-safe palette, so you can see the different continents easily. The countries furthest from the trend line are labelled directly for easy identification.

In [ ]:
# Q3
# YOUR CODE HERE

d = df[['Country', 'Continent', 'Area', '2022 Population']].copy()
d['logA'], d['logP'] = np.log10(d['Area']), np.log10(d['2022 Population'])
slope, intercept = np.polyfit(d['logA'], d['logP'], 1)
r = np.corrcoef(d['logA'], d['logP'])[0, 1]
d['resid'] = d['logP'] - (slope * d['logA'] + intercept)

above = (d['resid'] > 0).groupby(d['Continent']).mean().sort_values()

fig = px.scatter(d, x='Area', y='2022 Population', color='Continent', hover_name='Country',
                 color_discrete_map=CONT_COLORS, log_x=True, log_y=True,
                 category_orders={'Continent': list(CONT_COLORS)},
                 template='plotly_white', height=600)
fig.update_traces(marker=dict(size=9, opacity=0.8, line=dict(width=0.5, color='white')))

xs = np.logspace(d['logA'].min(), d['logA'].max(), 50)
fig.add_scatter(x=xs, y=10 ** (slope * np.log10(xs) + intercept), mode='lines',
                line=dict(color='#9E9E9E', dash='dash', width=2), showlegend=False, hoverinfo='skip')

labels_q3 = {'India': (0, -22), 'China': (0, -22), 'Bangladesh': (-40, -25), 'Singapore': (0, -22),
             'Macau': (0, -22), 'Greenland': (0, 24), 'Mongolia': (30, 28), 'Falkland Islands': (0, 24)}
for name, (ax, ay) in labels_q3.items():
    row = d[d['Country'] == name].iloc[0]
    fig.add_annotation(x=np.log10(row['Area']), y=np.log10(row['2022 Population']), text=name,
                       showarrow=True, arrowhead=0, arrowcolor='#9E9E9E', ax=ax, ay=ay,
                       font=dict(size=12, color=DARK))

fig.update_layout(
    title=dict(text=f"Bigger countries do have more people (r = {r:.2f} on log scales), but continents differ:<br>"
                    f"{above['Asia']:.0%} of Asian countries sit above the trend line, vs only {above['South America']:.0%} in South America and {above['Oceania']:.0%} in Oceania"),
    font=FONT, legend=dict(title=None, orientation='h', y=1.02, x=0, yanchor='bottom'),
    margin=dict(t=120))
fig.update_xaxes(title='Land area (km², log scale)', showgrid=False)
fig.update_yaxes(title='Population 2022 (log scale)')
fig.show()

---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

I chose the box plot with all countries shown as points because the question asks for the spread within and across continents, not just an average. A box plot shows the median, the middle 50% and the outliers for each continent side by side. With only 20–60 countries per continent, a box plot is clearer than a violin plot. Density uses a log scale because it ranges from under 1 to over 20,000 people per km². Continents are sorted by median and boxes are grey. The vermillion dashed line marks the world median, and the extreme countries are labelled.

In [16]:
# Q4
# YOUR CODE HERE

order = df.groupby('Continent')['Density'].median().sort_values().index.tolist()
world_med = df['Density'].median()
med = df.groupby('Continent')['Density'].median()

fig = px.box(df, x='Density', y='Continent', points='all', hover_name='Country',
             category_orders={'Continent': order}, log_x=True,
             template='plotly_white', height=560)
fig.update_traces(marker=dict(color='#0072B2', size=5, opacity=0.6), line=dict(color=DARK),
                  fillcolor='rgba(189,189,189,0.35)', jitter=0.4, pointpos=0)

fig.add_vline(x=np.log10(world_med), line_dash='dash', line_color=ACCENT,
              annotation=dict(text=f'World median: {world_med:.0f} people/km²', font=dict(color=ACCENT),
                              xanchor='left', yanchor='bottom', y=1, yref='paper'))

for name in ['Macau', 'Monaco', 'Singapore', 'Bangladesh', 'Greenland', 'Mongolia', 'Falkland Islands']:
    row = df[df['Country'] == name].iloc[0]
    fig.add_annotation(x=np.log10(row['Density']), y=row['Continent'], text=name, showarrow=True,
                       arrowhead=0, arrowcolor='#9E9E9E', ax=0, ay=-24, font=dict(size=11, color=DARK))

fig.update_layout(
    title=dict(text=f"Density varies far more within continents than between them: Asia ranges from "
                    f"{df.loc[df.Continent=='Asia','Density'].min():.0f} to {df.loc[df.Continent=='Asia','Density'].max():,.0f} people/km²<br>"
                    f"South America is the least crowded (median {med['South America']:.0f}); small islands push "
                    f"North America's median to {med['North America']:.0f}"),
    font=FONT, showlegend=False, margin=dict(t=110))
fig.update_xaxes(title='People per km² (2022, log scale)')
fig.update_yaxes(title=None, showgrid=False)
fig.show()

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

I chose the choropleth map because the question asks *where in the world* countries grew or shrank, so the data needs to be shown on a map. I grouped percentage change into classes instead of using a continuous scale, because the key split is grew vs shrank and a few extreme values like Qatar (+317%) would wash out the rest. The colours are diverging and CVD-safe, vermillion for shrinking and blues for growing, getting darker with faster growth. The legend stays because a map needs its colour key.

In [17]:
# Q5
# YOUR CODE HERE
df['Change'] = (df['2022 Population'] / df['2000 Population'] - 1) * 100
bins = [-np.inf, 0, 25, 50, 100, np.inf]
labels = ['Shrank', '+0–25%', '+25–50%', '+50–100%', 'More than doubled']
df['Change class'] = pd.cut(df['Change'], bins=bins, labels=labels).astype(str)

n_shrink = (df['Change'] < 0).sum()
shrink_eu = ((df['Change'] < 0) & (df['Continent'] == 'Europe')).sum()
n_double = (df['Change'] > 100).sum()
double_af = ((df['Change'] > 100) & (df['Continent'] == 'Africa')).sum()

fig = px.choropleth(df, locations='CCA3', locationmode='ISO-3', color='Change class',
                    hover_name='Country', hover_data={'Change': ':.1f', 'CCA3': False, 'Change class': False},
                    category_orders={'Change class': labels},
                    color_discrete_map={'Shrank': ACCENT, '+0–25%': '#C6DBEF', '+25–50%': '#6BAED6',
                                        '+50–100%': '#2171B5', 'More than doubled': '#08306B'},
                    template='plotly_white', height=560)
fig.update_traces(marker_line_color='white', marker_line_width=0.4)
fig.update_geos(projection_type='natural earth', showframe=False, showcoastlines=False,
                showcountries=False, showland=True, landcolor='#EEEEEE', lataxis_range=[-58, 85])
fig.update_layout(
    title=dict(text=f"<span style='color:{ACCENT}'><b>{n_shrink} countries shrank</b></span> between 2000 and 2022, "
                    f"{shrink_eu} of them in Europe (mostly the East),<br>while {double_af} of the {n_double} countries "
                    f"that more than doubled are in Africa"),
    font=FONT, margin=dict(l=0, r=0, t=90, b=0),
    legend=dict(title='Change 2000–2022', orientation='h', y=-0.02, x=0.5, xanchor='center'))
fig.show()

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

I chose the treemap because the question asks for a part-to-whole view at two levels, continents and the countries inside them. So, a treemap nests countries inside continents, and each rectangle's area is its share of the 2022 world population. A pie or bar chart can't show both levels at once. Colour encodes continent with a CVD-safe palette, and labels show each box's share of the world directly, so no legend is needed.

In [18]:
# Q6
# YOUR CODE HERE

total = df['2022 Population'].sum()
cont_share = df.groupby('Continent')['2022 Population'].sum() / total * 100
ci_share = df.loc[df['Country'].isin(['China', 'India']), '2022 Population'].sum() / total * 100

fig = px.treemap(df, path=[px.Constant('World'), 'Continent', 'Country'], values='2022 Population',
                 color='Continent', color_discrete_map={**CONT_COLORS, '(?)': '#E0E0E0'},
                 template='plotly_white', height=650)
fig.update_traces(texttemplate='<b>%{label}</b><br>%{percentRoot:.1%}', textfont=dict(size=14),
                  hovertemplate='<b>%{label}</b><br>%{value:,.0f} people<br>%{percentRoot:.1%} of world<extra></extra>',
                  marker=dict(line=dict(color='white', width=1)), root_color='#F5F5F5')
fig.update_layout(
    title=dict(text=f"Asia is home to {cont_share['Asia']:.0f}% of the world's {total/1e9:.1f} billion people; "
                    f"China and India alone hold {ci_share:.0f}%"),
    font=FONT, margin=dict(l=10, r=10, t=60, b=10))
fig.show()

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

I chose the 100% stacked bar chart becuase the question is about shares of a whole (world population = 100%) changing over time. A 100% stacked bar for each year shows the composition and how it shifts. I put Africa at the bottom and Europe at the top, so both sit on a fixed edge (0% and 100%) and are easy to compare across years. Africa, Asia and Europe each get their own colour-blind-safe colour (orange, green, blue), and the rest of the world is light grey context. Each segment shows its percentage, and the groups are labelled on the right instead of in a legend. Years are shown as categories because the gaps between them are uneven (10, 5 and 2 years).

In [26]:
# Q7
grp = df_long.copy()
grp['Group'] = np.where(grp['Continent'].isin(['Africa', 'Asia', 'Europe']), grp['Continent'], 'Rest of the world')
share = grp.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
share['Share'] = share['Population'] / share.groupby('Year')['Population'].transform('sum') * 100
share['Year'] = share['Year'].astype(str)

ASIA_GREEN = '#009E73'   # Okabe-Ito bluish green: distinguishable from vermillion for colour-blind readers
stack = ['Africa', 'Asia', 'Rest of the world', 'Europe']          # bottom -> top
colors = {'Africa': ACCENT, 'Asia': ASIA_GREEN, 'Rest of the world': '#E0E0E0', 'Europe': '#0072B2'}
s = share.pivot(index='Year', columns='Group', values='Share')

fig = px.bar(share, x='Year', y='Share', color='Group', category_orders={'Group': stack},
             color_discrete_map=colors, text=share['Share'].map(lambda v: f'{v:.0f}%'),
             template='plotly_white', height=560)
fig.update_traces(textposition='inside', insidetextanchor='middle', marker_line_width=0, textfont_size=12)
for g in ['Africa', 'Asia', 'Europe']:
    fig.update_traces(selector=dict(name=g), textfont_color='white')

# Direct labels to the right of the last bar instead of a legend
cum = 0
for g in stack:
    v = s.loc['2022', g]
    fig.add_annotation(x=len(s) - 1, y=cum + v / 2,   # x = index of the last bar
                       text=f'<b>{g}</b>' if g != 'Rest of the world' else g,
                       showarrow=False, xanchor='left', xshift=32,
                       font=dict(color=colors[g] if g != 'Rest of the world' else '#808080', size=13))
    cum += v

fig.update_layout(
    title=dict(text=f"<span style='color:{ACCENT}'><b>Africa's</b></span> share of the world rose from {s.loc['1970','Africa']:.0f}% to "
                    f"{s.loc['2022','Africa']:.0f}% since 1970, while <span style='color:#0072B2'><b>Europe's</b></span> "
                    f"halved from {s.loc['1970','Europe']:.0f}% to {s.loc['2022','Europe']:.0f}%;<br>"
                    f"<span style='color:{ASIA_GREEN}'><b>Asia</b></span> stayed steady at about {s['Asia'].mean():.0f}%"),
    font=FONT, showlegend=False, bargap=0.25, margin=dict(r=150, t=90))
fig.update_xaxes(title=None, showgrid=False, type='category')   # uneven year gaps -> categories
fig.update_yaxes(title=None, ticksuffix='%', range=[0, 100], showgrid=False)
fig.show()